# Designing and Evaluating a Modular Retrieval-Augmented Question Answering System

### Lexical, Dense, and Reciprocal Rank Fusion Retrieval with End-to-End Evaluation

## Overview

Retrieval-Augmented Generation (RAG) combines document retrieval with large language model generation to answer questions using external knowledge rather than relying solely on information stored in model parameters. The overall performance of a RAG system depends not only on the language model, but also on the quality of the retrieval pipeline and how retrieved evidence is incorporated into the prompt.

This project designs and evaluates a modular retrieval-augmented question-answering system using the Stanford Question Answering Dataset (SQuAD). The objective is to study how different retrieval strategies influence downstream answer generation while keeping the language model fixed.

The system follows the pipeline:

```text
Question
   │
   ▼
Preprocessing
   │
   ▼
Retriever
   ├── Lexical retrieval
   ├── Dense retrieval
   └── Reciprocal Rank Fusion
   │
   ▼
Prompt construction
   │
   ▼
Large language model
   │
   ▼
Answer
   │
   ▼
Evaluation
```

The project compares three retrieval strategies:

- **Lexical retrieval**, which ranks contexts according to exact word overlap between the question and candidate passages.
- **Dense retrieval**, which represents questions and passages using BERT embeddings and retrieves contexts based on cosine similarity.
- **Reciprocal Rank Fusion (RRF)**, which combines the rankings produced by the lexical and dense retrievers to exploit both lexical matching and semantic similarity.

The retrieved contexts are assembled into a prompt and provided to a large language model to generate answers.

The retrieval pipeline is evaluated through a series of progressively more sophisticated experiments, including:

- Vanilla question answering without retrieval.
- Oracle question answering using the ground-truth context.
- Lexical retrieval.
- Dense retrieval.
- The effect of varying the number of retrieved contexts.
- Hybrid retrieval using Reciprocal Rank Fusion.

End-to-end question-answering performance is evaluated using:

- Exact Match (EM)
- Token-level F1
- ROUGE-2 F1

The objective is to understand how different retrieval strategies influence answer quality and to investigate whether combining complementary retrieval signals through Reciprocal Rank Fusion improves downstream question-answering performance.

## 1. Data Preparation and Preprocessing

This section loads the SQuAD dataset and transforms it into a retrieval corpus and a reproducible question-answering benchmark.

The preprocessing pipeline:

1. Loads the raw SQuAD JSON data.
2. Extracts all context paragraphs to form the retrieval corpus.
3. Filters for answerable questions.
4. Stores each question with its gold answer and supporting context.
5. Creates separate development and test sets for experimentation and final evaluation.

In [35]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
!pip install -q \
    transformers \
    accelerate \
    sentence-transformers \
    rouge-score \
    rank-bm25

  Preparing metadata (setup.py) ... done


In [3]:
import collections
import copy
import json
import os
import random
import re
import string
import time
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

In [4]:
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Device: {DEVICE}")

if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: Tesla T4


In [5]:
DATA_DIR = Path("squad_data")
DATA_DIR.mkdir(exist_ok=True)

TRAIN_URL = (
    "https://rajpurkar.github.io/"
    "SQuAD-explorer/dataset/train-v2.0.json"
)

TRAIN_PATH = DATA_DIR / "squad_train.json"

if not TRAIN_PATH.exists():
    urlretrieve(TRAIN_URL, TRAIN_PATH)
    print("Downloaded SQuAD 2.0.")
else:
    print("SQuAD 2.0 already exists.")

Downloaded SQuAD 2.0.


In [6]:
# Load the raw SQuAD dataset
with open(TRAIN_PATH, "r", encoding="utf-8") as file:
    train_data = json.load(file)

# Inspect one representative example
sample_topic = train_data["data"][0]
sample_paragraph = sample_topic["paragraphs"][0]
sample_qa = sample_paragraph["qas"][0]

print(f"Topics: {len(train_data['data']):,}")
print(f"Sample topic: {sample_topic['title']}")
print(f"Paragraphs in sample topic: {len(sample_topic['paragraphs'])}")
print(f"\nSample context:\n{sample_paragraph['context'][:500]}...")
print(f"\nSample question: {sample_qa['question']}")
print(f"Sample answer: {sample_qa['answers'][0]['text']}")

Topics: 442
Sample topic: Beyoncé
Paragraphs in sample topic: 66

Sample context:
Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ bee-YON-say) (born September 4, 1981) is an American singer, songwriter, record producer and actress. Born and raised in Houston, Texas, she performed in various singing and dancing competitions as a child, and rose to fame in the late 1990s as lead singer of R&B girl-group Destiny's Child. Managed by her father, Mathew Knowles, the group became one of the world's best-selling girl groups of all time. Their hiatus saw the release of Beyoncé's debut al...

Sample question: When did Beyonce start becoming popular?
Sample answer: in the late 1990s


In [7]:
print("Total number of paragraphs in the training set:", sum([len(topic['paragraphs']) for topic in train_data['data']]))
print("Total number of question-answer pairs in the training set:", sum([len(paragraph['qas']) for topic in train_data['data'] for paragraph in topic['paragraphs']]))

Total number of paragraphs in the training set: 19035
Total number of question-answer pairs in the training set: 130319


In [8]:
# SQuAD 2.0 includes both answerable and unanswerable questions.
# This project focuses on retrieval and grounded answer generation, so the benchmark will use only questions with at least one annotated answer.

print("Avg number of answers per question:",
      sum([len(qa['answers']) for topic in train_data['data'] for paragraph in topic['paragraphs'] for qa in paragraph['qas']]) /
      sum([len(paragraph['qas']) for topic in train_data['data'] for paragraph in topic['paragraphs']]))
print("Count of answerable vs unanswerable questions:")
answerable_count = 0
unanswerable_count = 0
for topic in train_data['data']:
    for paragraph in topic['paragraphs']:
        for qa in paragraph['qas']:
            if len(qa['answers']) > 0:
                answerable_count += 1
            else:
                unanswerable_count += 1
print(f"Answerable questions: {answerable_count} ({answerable_count / (answerable_count + unanswerable_count) * 100:.2f}%)")
print(f"Unanswerable questions: {unanswerable_count} ({unanswerable_count / (answerable_count + unanswerable_count) * 100:.2f}%)")

Avg number of answers per question: 0.6662190471074824
Count of answerable vs unanswerable questions:
Answerable questions: 86821 (66.62%)
Unanswerable questions: 43498 (33.38%)


In [9]:
# Finally, create the RAG QA benchmark consisting of 250 answerable questions.

# We will use all available context paragraphs for RAG
rag_contexts = [paragraph['context'] for topic in train_data['data'] for paragraph in topic['paragraphs']]

qa_pairs = []
for topic in train_data['data']:
    for paragraph in topic['paragraphs']:
        for qa in paragraph['qas']:
            if len(qa['answers']) > 0:
                qa_pairs.append({
                    "question": qa['question'],
                    "answer": qa['answers'][0]['text'],
                    "context": paragraph['context']
                })

# randomly sample 250 answerable questions for the benchmark
import random
random.seed(42)
sampled_qa_pairs = random.sample(qa_pairs, 250)


evaluation_benchmark = {'qas': sampled_qa_pairs,
                        'contexts': rag_contexts}

# save the evaluation benchmark to a file
json.dump(evaluation_benchmark, open(f"{DATA_DIR}/rag_qa_benchmark.json", "w"), indent=2)

In [10]:
# load the benchmark and display some samples
evaluation_benchmark = json.load(open(f"{DATA_DIR}/rag_qa_benchmark.json"))

print("Sample RAG contexts:")
for context in evaluation_benchmark['contexts'][:2]:
    print(context)
    print("-"*20)
print("="*30)
print("Sample RAG QA pairs:")
for qa in evaluation_benchmark['qas'][:5]:
    print(f"Question: {qa['question']}")
    print(f"Answer: {qa['answer']}")
    print("-"*20)

Sample RAG contexts:
Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ bee-YON-say) (born September 4, 1981) is an American singer, songwriter, record producer and actress. Born and raised in Houston, Texas, she performed in various singing and dancing competitions as a child, and rose to fame in the late 1990s as lead singer of R&B girl-group Destiny's Child. Managed by her father, Mathew Knowles, the group became one of the world's best-selling girl groups of all time. Their hiatus saw the release of Beyoncé's debut album, Dangerously in Love (2003), which established her as a solo artist worldwide, earned five Grammy Awards and featured the Billboard Hot 100 number-one singles "Crazy in Love" and "Baby Boy".
--------------------
Following the disbandment of Destiny's Child in June 2005, she released her second solo album, B'Day (2006), which contained hits "Déjà Vu", "Irreplaceable", and "Beautiful Liar". Beyoncé also ventured into acting, with a Golden Globe-nominated performance in Dre

The `evaluation_benchmark` is a dictionary with two keys:
* `evaluation_benchmark['qas']`  provides a list of *qa_items* (see below).
* `evaluation_benchmark['contexts']` provides a list of available candidate contexts. Note that this includes all contexts from SQuAD, not just the ones for the 250 questions we sampled for the benchmark.

Each *qa_item* is a dictionary with the following keys:
* `qa_item['question']` is the question string
* `qa_item['answer']` is the target answer string
* `qa_item['context']` is the gold context for this question

For example:


In [11]:
qa_items = evaluation_benchmark['qas']
len(qa_items)

250

In [12]:
qa_item = qa_items[0]
qa_item['question']

'Which which kingdoms did the taifas establish diplomatic relations?'

In [13]:
qa_item['answer']

'the Christian kingdoms of the north'

In [14]:
qa_item['context']

'The governors of the taifas each proclaimed themselves Emir of their provinces and established diplomatic relations with the Christian kingdoms of the north. Most of Portugal fell into the hands of the Taifa of Badajoz of the Aftasid Dynasty, and after a short spell of an ephemeral Taifa of Lisbon in 1022, fell under the dominion of the Taifa of Seville of the Abbadids poets. The Taifa period ended with the conquest of the Almoravids who came from Morocco in 1086 winning a decisive victory at the Battle of Sagrajas, followed a century later in 1147, after the second period of Taifa, by the Almohads, also from Marrakesh.'

## Part 1 — Question Answering Evaluation Metrics

This section defines the metrics used to compare a generated answer with the corresponding reference answer.

Because several of the metrics operate at the token level, both answers are first normalized by:

- converting text to lowercase;
- removing punctuation;
- removing articles such as `a`, `an`, and `the`;
- removing extra whitespace.

The normalized answers are then compared using Exact Match, token-level F1, and ROUGE.

In [15]:
def normalize_answer(s):
  """Lower text and remove punctuation, articles and extra whitespace."""
  def remove_articles(text):
    regex = re.compile(r'\b(a|an|the)\b', re.UNICODE)
    return re.sub(regex, ' ', text)
  def white_space_fix(text):
    return ' '.join(text.split())
  def remove_punc(text):
    exclude = set(string.punctuation)
    return ''.join(ch for ch in text if ch not in exclude)
  def lower(text):
    return text.lower()
  return white_space_fix(remove_articles(remove_punc(lower(s))))

def get_tokens(s):
  if not s: return []
  return normalize_answer(s).split()

First, Exact Match (EM) measures the percentage of predictions that match any one of the ground truth answers exactly after normalization.
The following function returns 1 if the predicted answer is correct and 0 otherwise.

In [16]:
def compute_exact(a_gold, a_pred):
  return int(normalize_answer(a_gold) == normalize_answer(a_pred))

### Token-Level F1 Score

Unlike Exact Match, which requires a prediction to be identical to the reference answer, the **F1 score** measures **partial overlap** between the predicted and reference answers. It rewards answers that contain many of the correct tokens, even if they are not an exact match.

The F1 score is the harmonic mean of **precision** and **recall**:

$$
F_1 = \frac{2 \times \text{Precision} \times \text{Recall}}
{\text{Precision} + \text{Recall}}
$$

where

- **Precision** measures how many predicted tokens are correct:

$$
\text{Precision} =
\frac{\text{# overlapping tokens}}
{\text{# predicted tokens}}
$$

- **Recall** measures how much of the reference answer was recovered:

$$
\text{Recall} =
\frac{\text{# overlapping tokens}}
{\text{# reference tokens}}
$$

A perfect prediction receives an F1 score of **1**, while a prediction with no overlapping tokens receives **0**.

For question answering, F1 is often more informative than Exact Match because it gives partial credit when a prediction captures most of the correct answer but differs slightly in wording.

In [17]:
from collections import Counter

def compute_f1(a_gold, a_pred):
    gold_tokens = normalize_answer(a_gold).split()
    pred_tokens = normalize_answer(a_pred).split()

    common = Counter(gold_tokens) & Counter(pred_tokens)
    num_same = sum(common.values())

    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)

    f1 = 2 * precision * recall / (precision + recall)
    return f1

In [18]:
# Testing the function

gold = "The University of Oxford"

predictions = [
    "Oxford",
    "University of Oxford",
    "The University of Oxford",
    "Cambridge University"
]

for pred in predictions:
    print(f"Prediction: {pred}")
    print(f"EM: {compute_exact(gold, pred)}")
    print(f"F1: {compute_f1(gold, pred):.3f}")
    print("-" * 40)

Prediction: Oxford
EM: 0
F1: 0.500
----------------------------------------
Prediction: University of Oxford
EM: 1
F1: 1.000
----------------------------------------
Prediction: The University of Oxford
EM: 1
F1: 1.000
----------------------------------------
Prediction: Cambridge University
EM: 0
F1: 0.400
----------------------------------------


Finally, we are also want to compute ROUGE-2 scores (which extends the F1 score above to 2-grams). We can use the `rouge_score` package to do this for us.

In [19]:
from rouge_score import rouge_scorer

rouge_scorer = rouge_scorer.RougeScorer(['rouge2'], use_stemmer=False)

def compute_rouge2(a_gold, a_pred):
    if not a_gold or not a_pred:
        return 0.0
    scores = rouge_scorer.score(a_gold.lower(), a_pred.lower())
    return scores['rouge2'].fmeasure

### Why Use Multiple Evaluation Metrics?

No single metric fully captures answer quality, so we evaluate our QA system using **three complementary metrics**, each measuring a different aspect of correctness.

- **Exact Match (EM)** measures **strict correctness**. A prediction receives a score of **1** only if it exactly matches the reference answer after normalization, and **0** otherwise.

- **Token-level F1** measures **word-level overlap** between the prediction and the reference answer. It gives partial credit when the prediction captures much of the correct answer but is incomplete or contains additional words.

- **ROUGE-2** measures **bigram overlap** (pairs of consecutive words). Unlike F1, it considers local word order and phrase structure, making it more sensitive to whether the prediction preserves the phrasing of the reference answer.


#### Example

Suppose the reference answer is:

> **New York City**

| Prediction | Exact Match | F1 | ROUGE-2 | Explanation |
|------------|:-----------:|:--:|:--------:|-------------|
| **New York City** | ✅ | 1.00 | 1.00 | Perfect prediction. |
| **New York** | ❌ | High | Moderate | Most of the answer is correct, but it is incomplete. |
| **York New City** | ❌ | High | Very Low | Contains the same words but in the wrong order, so the phrase structure is lost. |

This example illustrates why relying on a single metric can be misleading. Exact Match is often too strict, F1 captures semantic overlap but ignores word order, and ROUGE-2 evaluates whether the model preserves meaningful phrases. Using all three provides a balanced assessment of answer quality.

Let's test the metrics:

In [20]:
reference_answers = ["London", "The capital of England is London.", "London is the capital city of England."]
predicted_answers = ["London, capital of England"] * len(reference_answers)

print("Normalized Answers:")
for ref, pred in zip(reference_answers, predicted_answers):
    print(f"Original:")
    print(f"Reference: {ref} | Predicted: {pred}")
    print(f"Normalized:")
    print(f"Reference: {normalize_answer(ref)} | Predicted: {normalize_answer(pred)}")
    print("Exact Match:", compute_exact(normalize_answer(ref), normalize_answer(pred)))
    print("F1 Score:", compute_f1(normalize_answer(ref), normalize_answer(pred)))
    print("ROUGE-2 F1-score:", compute_rouge2(normalize_answer(ref), normalize_answer(pred)))
    print("-"*40)

Normalized Answers:
Original:
Reference: London | Predicted: London, capital of England
Normalized:
Reference: london | Predicted: london capital of england
Exact Match: 0
F1 Score: 0.4
ROUGE-2 F1-score: 0.0
----------------------------------------
Original:
Reference: The capital of England is London. | Predicted: London, capital of England
Normalized:
Reference: capital of england is london | Predicted: london capital of england
Exact Match: 0
F1 Score: 0.888888888888889
ROUGE-2 F1-score: 0.5714285714285715
----------------------------------------
Original:
Reference: London is the capital city of England. | Predicted: London, capital of England
Normalized:
Reference: london is capital city of england | Predicted: london capital of england
Exact Match: 0
F1 Score: 0.8
ROUGE-2 F1-score: 0.25
----------------------------------------


# Part 2 — Vanilla Question Answering

In this section, we establish a **baseline** by evaluating a pretrained large language model (LLM) on the benchmark **without providing any external context**. The model must answer each question using only the knowledge acquired during pretraining.

To simplify inference, we use the Hugging Face `pipeline` API, a high-level interface that abstracts away much of the boilerplate required to interact with a language model. Given a text prompt, the pipeline automatically:

1. **Tokenizes** the input by converting the text into numerical tokens that the model can process.
2. **Runs inference** by feeding the tokens through the pretrained model to generate a response.
3. **Decodes** the generated tokens back into human-readable text, removing special tokens such as end-of-sequence markers.

This allows us to interact with the model using a single function call, without manually handling the underlying preprocessing and postprocessing steps.

## Loading the Language Model

We use the **OLMo-2 1B Instruct** model, an instruction-tuned open-source language model developed by Allen AI. The model is loaded through the Hugging Face `transformers` library, which automatically downloads the model weights and tokenizer on first use.

In [21]:
qa_model = "allenai/OLMo-2-0425-1B-Instruct"

from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=qa_model,
    dtype=torch.bfloat16,
    device_map=DEVICE,
)

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.97GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/179 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/4.88k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.14M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/581 [00:00<?, ?B/s]

We can now pass a prompt to the model and retreive the completed answer.

In [21]:
prompt = "My favorite thing to do in fall is"
output = pipe(prompt,
              max_new_tokens=128,
              do_sample=True,
              pad_token_id=pipe.tokenizer.eos_token_id)
print(output)

[transformers] Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[{'generated_text': 'My favorite thing to do in fall is to create a bouquet of autumn leaves, mums, and pumpkins in a vase on a table in our kitchen. The fragrance and soft colors make it a delightful and relaxing activity.\n \nAs the leaves begin to change, I like to take a walk in the autumn woods. The crisp air and vibrant colors provide the perfect backdrop for enjoying nature and taking in the tranquility of the season.\n \nSome favorite fall foods include roasted butternut squash and pumpkins, apple cider, cinnamon rolls with fallen leaves, and pumpkin spice lattes. Eating these delights while enjoying the season’s bounty is a great way to embrace the fall spirit'}]


We can skip the prompt that is repeated in the output.

In [22]:
output[0]['generated_text'][len(prompt):].strip()

'to create a bouquet of autumn leaves, mums, and pumpkins in a vase on a table in our kitchen. The fragrance and soft colors make it a delightful and relaxing activity.\n \nAs the leaves begin to change, I like to take a walk in the autumn woods. The crisp air and vibrant colors provide the perfect backdrop for enjoying nature and taking in the tranquility of the season.\n \nSome favorite fall foods include roasted butternut squash and pumpkins, apple cider, cinnamon rolls with fallen leaves, and pumpkin spice lattes. Eating these delights while enjoying the season’s bounty is a great way to embrace the fall spirit'

## Using the LLM for Question Answering

We now define a `vanilla_qa(qa_item)` function that generates an answer using only the pretrained model's internal knowledge.

The function:

1. extracts the question from `qa_item`;
2. inserts the question into a concise instruction prompt;
3. sends the prompt to the language model;
4. extracts and returns the generated answer as a string.

Importantly, the model receives **only the question**. It is not given the reference answer or the associated context, since this section is intended to measure the model's performance without retrieval.

An example prompt is:

> Answer the following question concisely. Return only the answer and do not provide an explanation.
>
> Question: Who played the lead role in *Alien*?
>
> Answer:

The prompt should encourage the model to produce a short, direct response, since the generated answer will be compared with the reference answer using Exact Match, F1, and ROUGE-2.

In [22]:
pipe.model.generation_config.max_length = None
pipe.model.generation_config.max_new_tokens = 32
pipe.model.generation_config.do_sample = False
pipe.model.generation_config.pad_token_id = pipe.tokenizer.eos_token_id

In [23]:
def vanilla_qa(qa_item):
    question = qa_item["question"]

    prompt = f"""Answer each question using the shortest possible answer.

Do not explain.
Do not repeat the question.
Return only the answer phrase.

Example:
Question: Who wrote Pride and Prejudice?
Answer: Jane Austen

Question: {question}
Answer:"""

    output = pipe(
        prompt,
        max_new_tokens=16,
        do_sample=False,
        pad_token_id=pipe.tokenizer.eos_token_id,
    )

    answer = output[0]["generated_text"][len(prompt):].strip()

    # Keep only the first generated line
    answer = answer.split("\n")[0].strip()

    return answer

The following code should return an answer (but possibly not the right one) to the first question in the dataset.

In [26]:
qa_item = evaluation_benchmark['qas'][0]
qa_item['question']

'Which which kingdoms did the taifas establish diplomatic relations?'

In [27]:
vanilla_qa(qa_item) # inspect the item

[transformers] Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'Visigothic Kingdom, Umayyad Caliphate, and the Kingdom'

And the following function evaluates the performance of the `vanilla_qa` function on a list of qa_items.

In [24]:
from transformers.utils import logging
logging.set_verbosity_error()

pipe.model.generation_config.max_length = None

In [25]:
def evaluate_qa(qa_function, qa_items, verbose=False):
    results = []

    for i, qa_item in tqdm(
        enumerate(qa_items),
        desc="Evaluating QA instances",
        total=len(qa_items)
    ):
        question = qa_item["question"]
        answer = qa_item["answer"]
        context = qa_item["context"]

        predicted_answer = qa_function(qa_item)

        exact_match = compute_exact(answer, predicted_answer)
        f1_score = compute_f1(answer, predicted_answer)
        rouge2_f1 = compute_rouge2(answer, predicted_answer)

        if verbose:
            print(f"Q: {question}")
            print(f"Gold Answer: {answer}")
            print(f"Predicted Answer: {predicted_answer}")
            print(f"Exact Match: {exact_match}, F1 Score: {f1_score:.3f}")
            print(f"ROUGE-2 F1 Score: {rouge2_f1:.3f}")
            print("-" * 40)

        results.append({
            "question": question,
            "answer": answer,
            "predicted_answer": predicted_answer,
            "context": context if context else None,
            "exact_match": exact_match,
            "f1_score": f1_score,
            "rouge2_f1": rouge2_f1
        })

    return results

In [30]:
vanilla_evaluation_results = evaluate_qa(vanilla_qa, evaluation_benchmark['qas'])

Evaluating QA instances:   0%|          | 0/250 [00:00<?, ?it/s]

The function returns a list of evaluation results, one dictionary for each qa item.

In [31]:
vanilla_evaluation_results[0]

{'question': 'Which which kingdoms did the taifas establish diplomatic relations?',
 'answer': 'the Christian kingdoms of the north',
 'predicted_answer': 'Visigothic Kingdom, Umayyad Caliphate, and the Kingdom',
 'context': 'The governors of the taifas each proclaimed themselves Emir of their provinces and established diplomatic relations with the Christian kingdoms of the north. Most of Portugal fell into the hands of the Taifa of Badajoz of the Aftasid Dynasty, and after a short spell of an ephemeral Taifa of Lisbon in 1022, fell under the dominion of the Taifa of Seville of the Abbadids poets. The Taifa period ended with the conquest of the Almoravids who came from Morocco in 1086 winning a decisive victory at the Battle of Sagrajas, followed a century later in 1147, after the second period of Taifa, by the Almohads, also from Marrakesh.',
 'exact_match': 0,
 'f1_score': 0.0,
 'rouge2_f1': 0.0}

Finally, the `present_results` function aggregates the results for the various qa items and prints the overall result.

In [26]:
def present_results(eval_results, exp_name=""):
    print(f"{exp_name} Evaluation Results:")
    exact_matches = [res['exact_match'] for res in eval_results]
    f1_scores = [res['f1_score'] for res in eval_results]
    rouge2_f1 = [res['rouge2_f1'] for res in eval_results]
    print(f"Exact Match: {sum(exact_matches) / len(exact_matches) * 100:.2f}%")
    print(f"F1 Score: {sum(f1_scores) / len(f1_scores) * 100:.2f}%")
    print(f"ROUGE2 F1: {sum(rouge2_f1) / len(rouge2_f1) * 100:.2f}%")

    # print out some evaluation results
    for res in eval_results[:5]:
        print(f"Question: {res['question']}")
        print(f"Gold Answer: {res['answer']}")
        print(f"Predicted Answer: {res['predicted_answer']}")
        print(f"Exact Match: {res['exact_match']}, F1 Score: {res['f1_score']}")
        print("ROUGE-2 F1-score:", res['rouge2_f1'])
        print("-"*40)

In [33]:
present_results(vanilla_evaluation_results, "Vanilla QA")

Vanilla QA Evaluation Results:
Exact Match: 8.00%
F1 Score: 14.04%
ROUGE2 F1: 3.11%
Question: Which which kingdoms did the taifas establish diplomatic relations?
Gold Answer: the Christian kingdoms of the north
Predicted Answer: Visigothic Kingdom, Umayyad Caliphate, and the Kingdom
Exact Match: 0, F1 Score: 0.0
ROUGE-2 F1-score: 0.0
----------------------------------------
Question: When was Nanjing extended?
Gold Answer: 211 AD
Predicted Answer: 1936
Exact Match: 0, F1 Score: 0.0
ROUGE-2 F1-score: 0.0
----------------------------------------
Question: In what district of Manhattan were the Occupy Wall Street protests?
Gold Answer: Financial District
Predicted Answer: Zuccotti Park
Exact Match: 0, F1 Score: 0.0
ROUGE-2 F1-score: 0.0
----------------------------------------
Question: Illyrian pirates caused two wars in the Balkans with what Empire?
Gold Answer: Roman
Predicted Answer: Rome
Exact Match: 0, F1 Score: 0.0
ROUGE-2 F1-score: 0.0
----------------------------------------
Ques

# Part 3 — Oracle Question Answering

The vanilla QA baseline evaluated the language model's ability to answer questions using only the knowledge stored in its parameters. We now establish an **oracle** baseline by providing the model with the **gold context** associated with each question.

For every question, the corresponding context is taken directly from the evaluation benchmark. Since this context contains the information needed to answer the question, retrieval errors are completely eliminated. As a result, this experiment approximates the best performance that a retrieval-augmented question answering system could achieve if it always retrieved the correct document.

The `oracle_qa(qa_item)` function follows the same interface as `vanilla_qa`, but constructs the prompt using both the question and its associated context. The generated answer can therefore be evaluated using the same metrics (Exact Match, F1, and ROUGE-2), allowing for a direct comparison between the closed-book baseline and a system with perfect retrieval.

A typical prompt has the following structure:

> Answer the question using only the context below. Return only the shortest answer phrase. Do not explain.
>
> **Context:**  
> *[Gold context]*
>
> **Question:**  
> *[Question]*
>
> **Answer:**

Comparing the performance of the vanilla and oracle systems quantifies the value of providing relevant external knowledge to the language model. The remaining gap between the oracle system and a full retrieval-augmented pipeline will later reflect the quality of the retrieval component itself.

In [27]:
def oracle_qa(qa_item):
    question = qa_item["question"]
    context = qa_item["context"]

    prompt = f"""Answer the question using only the context below.

Return only the shortest answer phrase.
Do not explain.
Do not repeat the question.

Context:
{context}

Question: {question}

Answer:"""

    output = pipe(
        prompt,
        max_new_tokens=16,
        do_sample=False,
        pad_token_id=pipe.tokenizer.eos_token_id,
    )

    answer = output[0]["generated_text"][len(prompt):].strip()

    # Prevent the model from continuing with an explanation
    answer = answer.split("\n")[0].strip()

    return answer

In [35]:
oracle_evaluation_results = evaluate_qa(oracle_qa, evaluation_benchmark['qas'])
present_results(oracle_evaluation_results)

Evaluating QA instances:   0%|          | 0/250 [00:00<?, ?it/s]

 Evaluation Results:
Exact Match: 58.00%
F1 Score: 72.13%
ROUGE2 F1: 39.61%
Question: Which which kingdoms did the taifas establish diplomatic relations?
Gold Answer: the Christian kingdoms of the north
Predicted Answer: Christian kingdoms of the north
Exact Match: 1, F1 Score: 1.0
ROUGE-2 F1-score: 0.888888888888889
----------------------------------------
Question: When was Nanjing extended?
Gold Answer: 211 AD
Predicted Answer: 211 AD
Exact Match: 1, F1 Score: 1.0
ROUGE-2 F1-score: 1.0
----------------------------------------
Question: In what district of Manhattan were the Occupy Wall Street protests?
Gold Answer: Financial District
Predicted Answer: Financial District
Exact Match: 1, F1 Score: 1.0
ROUGE-2 F1-score: 1.0
----------------------------------------
Question: Illyrian pirates caused two wars in the Balkans with what Empire?
Gold Answer: Roman
Predicted Answer: Rome
Exact Match: 0, F1 Score: 0.0
ROUGE-2 F1-score: 0.0
----------------------------------------
Question: How 

# Part 4 — Retrieval-Augmented Question Answering

The previous experiments established two reference points for our question answering system. The **vanilla** baseline measured the model's ability to answer questions using only its pretrained knowledge, while the **oracle** baseline estimated the upper bound achievable when the correct context is provided.

We now introduce the retrieval component of a Retrieval-Augmented Generation (RAG) pipeline. Rather than supplying the gold context directly, the system must first identify the most relevant passages from a collection of candidate contexts before generating an answer.

As a first retrieval strategy, we implement a simple **token overlap retriever**. Given a question, the retriever measures lexical similarity by counting how many unique words are shared between the question and each candidate context. The contexts are then ranked according to this overlap score, and the top-*k* most relevant contexts are returned.

Although this approach is computationally inexpensive and straightforward to implement, it relies entirely on exact word matching. Consequently, it cannot recognize synonyms, paraphrases, or semantic similarity between different expressions. Nevertheless, it provides a useful baseline against which more sophisticated retrieval methods can later be compared.

In [51]:
candidate_contexts = evaluation_benchmark["contexts"]

In [52]:
len(candidate_contexts)

19035

In [53]:
candidate_contexts[0]

'Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ bee-YON-say) (born September 4, 1981) is an American singer, songwriter, record producer and actress. Born and raised in Houston, Texas, she performed in various singing and dancing competitions as a child, and rose to fame in the late 1990s as lead singer of R&B girl-group Destiny\'s Child. Managed by her father, Mathew Knowles, the group became one of the world\'s best-selling girl groups of all time. Their hiatus saw the release of Beyoncé\'s debut album, Dangerously in Love (2003), which established her as a solo artist worldwide, earned five Grammy Awards and featured the Billboard Hot 100 number-one singles "Crazy in Love" and "Baby Boy".'

### Token Overlap Retriever
Let's first experiment with a simple retriever based on word overlap. Given a question, we measure how many of its tokens appear in each of the candidate contexts. We then retrieve the k contexts with the highest overlap.

The function `retrieve_overlap(question, contexts, top_k)` takes in the question (a string) and a list of contexts (each context is a string) and calculates the word overlap between the question and *each* context, and return a list of the *top_k* contexts with the highest overlap.

In [54]:
def retrieve_overlap(question, contexts, top_k=5):
    question_tokens = set(get_tokens(question))

    scored_contexts = []

    for context in contexts:
        context_tokens = set(get_tokens(context))

        overlap = len(question_tokens & context_tokens)

        scored_contexts.append((overlap, context))

    scored_contexts.sort(key=lambda x: x[0], reverse=True)

    return [context for _, context in scored_contexts[:top_k]]

## Retrieving Candidate Contexts

The retrieval function is now applied to every question in the evaluation benchmark. For each question, the retriever ranks all candidate contexts according to their token overlap with the question and returns the top-*k* most relevant passages.

These retrieved contexts are stored in a new `rag_contexts` field for each question, creating a dataset that can be used to evaluate the complete retrieval-augmented question answering pipeline.

The following function runs the retriever a list of qa_items. For each qa_item it obtains the list of retrieved contexts and adds them to the qa_item.

In [55]:
def add_rag_context(qa_items, contexts, retriever, top_k=5):
    result_items = copy.deepcopy(qa_items)
    for inst in tqdm(result_items, desc="Retrieving contexts"):
        question = inst['question']
        retrieved_contexts = retriever(question, contexts, top_k)
        inst['rag_contexts'] = retrieved_contexts
    return result_items

In [56]:
overlap_rag_qa_pairs_top20 = add_rag_context(evaluation_benchmark["qas"], candidate_contexts, retrieve_overlap, top_k=20)

Retrieving contexts:   0%|          | 0/250 [00:00<?, ?it/s]

In [58]:
torch.save(overlap_rag_qa_pairs_top20, "/content/drive/MyDrive/rag_qa_evaluation/overlap_rag_qa_pairs_top20.pt")

In [59]:
overlap_rag_qa_pairs_top20 = torch.load("/content/drive/MyDrive/rag_qa_evaluation/overlap_rag_qa_pairs_top20.pt", weights_only=False)

In [60]:
def select_top_k_contexts(qa_items, top_k):
    result_items = copy.deepcopy(qa_items)

    for item in result_items:
        item["rag_contexts"] = item["rag_contexts"][:top_k]

    return result_items

In [61]:
overlap_datasets = {
    k: select_top_k_contexts(
        overlap_rag_qa_pairs_top20,
        top_k=k
    )
    for k in [1, 5, 10, 20]
}

In [62]:
rag_qa_pairs = overlap_datasets[5]

This returns a copy of the qa_item list that is now annotated with the additional 'rag_contexts'.

In [63]:
rag_qa_pairs[0]

{'question': 'Which which kingdoms did the taifas establish diplomatic relations?',
 'answer': 'the Christian kingdoms of the north',
 'context': 'The governors of the taifas each proclaimed themselves Emir of their provinces and established diplomatic relations with the Christian kingdoms of the north. Most of Portugal fell into the hands of the Taifa of Badajoz of the Aftasid Dynasty, and after a short spell of an ephemeral Taifa of Lisbon in 1022, fell under the dominion of the Taifa of Seville of the Abbadids poets. The Taifa period ended with the conquest of the Almoravids who came from Morocco in 1086 winning a decisive victory at the Battle of Sagrajas, followed a century later in 1147, after the second period of Taifa, by the Almohads, also from Marrakesh.',
 'rag_contexts': ["Although Turkey and Israel did not establish full diplomatic relations until 1991, Turkey has cooperated with the State since its recognition of Israel in 1949. Turkey's ties to the other Muslim-majority 

Before we run an end-to-end evaluation, we can check the accuracy of the word overlap retriever. In other words, for what fraction of questions is the gold context included in the top-k retrieved contexts.

In [64]:
# evaluation metric of retriever
def evaluate_retriever(rag_qa_pairs):
    """
    Evaluates the retriever by computing the accuracy of retrieved contexts against reference contexts.
    """
    correct_retrievals = 0
    for qa_item in rag_qa_pairs:
        if qa_item['context'] in qa_item['rag_contexts']:
            correct_retrievals += 1
    accuracy = correct_retrievals / len(rag_qa_pairs)
    return accuracy

In [65]:
evaluate_retriever(rag_qa_pairs)

0.532

## Retrieval-Augmented Question Answering

We now evaluate the complete retrieval-augmented question answering pipeline. Unlike the oracle system, which receives the correct gold context, the RAG system must answer using only the contexts selected by the overlap retriever.

For each question, the retrieved passages stored in `rag_contexts` are combined into a single prompt and provided to the language model. The model is instructed to rely only on these passages and return a concise answer.

This experiment measures the combined effect of retrieval quality and answer generation. Even when the language model is capable of extracting the correct answer, the system can only succeed if the retriever includes the relevant information among the top-*k* contexts.

In [66]:
def rag_qa(qa_item):
    question = qa_item["question"]
    retrieved_contexts = qa_item["rag_contexts"]

    # Combine the top-k retrieved passages into one context block
    context = "\n\n".join(
        f"Context {i + 1}:\n{retrieved_context}"
        for i, retrieved_context in enumerate(retrieved_contexts)
    )

    prompt = f"""Answer the question using only the retrieved contexts below.

Return only the shortest answer phrase.
Do not explain.
Do not repeat the question.
If the answer is not contained in the contexts, return "Unknown".

Retrieved contexts:
{context}

Question: {question}

Answer:"""

    output = pipe(
        prompt,
        max_new_tokens=16,
        do_sample=False,
        pad_token_id=pipe.tokenizer.eos_token_id,
    )

    answer = output[0]["generated_text"][len(prompt):].strip()

    # Prevent the model from continuing with an explanation
    answer = answer.split("\n")[0].strip()

    return answer

In [67]:
rag_overlap_eval = evaluate_qa(rag_qa, rag_qa_pairs)
present_results(rag_overlap_eval)

Evaluating QA instances:   0%|          | 0/250 [00:00<?, ?it/s]

 Evaluation Results:
Exact Match: 27.20%
F1 Score: 37.40%
ROUGE2 F1: 20.00%
Question: Which which kingdoms did the taifas establish diplomatic relations?
Gold Answer: the Christian kingdoms of the north
Predicted Answer: Portugal
Exact Match: 0, F1 Score: 0.0
ROUGE-2 F1-score: 0.0
----------------------------------------
Question: When was Nanjing extended?
Gold Answer: 211 AD
Predicted Answer: Nanjing ( listen; Chinese: 南京, "Southern Capital") is
Exact Match: 0, F1 Score: 0.0
ROUGE-2 F1-score: 0.0
----------------------------------------
Question: In what district of Manhattan were the Occupy Wall Street protests?
Gold Answer: Financial District
Predicted Answer: Lower Manhattan
Exact Match: 0, F1 Score: 0.0
ROUGE-2 F1-score: 0.0
----------------------------------------
Question: Illyrian pirates caused two wars in the Balkans with what Empire?
Gold Answer: Roman
Predicted Answer: Illyrian pirates
Exact Match: 0, F1 Score: 0.0
ROUGE-2 F1-score: 0.0
------------------------------------

# Part 5 — Retrieval-Augmented Question Answering with Dense Retrieval

The token overlap retriever introduced in the previous section relies on exact word matching to identify relevant contexts. While simple and efficient, this approach struggles when the question and the relevant passage use different wording to express the same idea.

To address this limitation, we now consider **dense retrieval**, where both questions and contexts are represented as continuous vector embeddings. Instead of comparing words directly, retrieval is based on the semantic similarity between these embeddings. As a result, passages that convey the same meaning using different vocabulary can still be identified as relevant.

We generate these embeddings using the pretrained **BERT-base-uncased** model. Each input sequence is tokenized and passed through BERT to produce contextualized representations for every token. Rather than using the representation of the special `[CLS]` token, we compute the embedding of the entire sequence by averaging the final hidden representations of all tokens (mean pooling), producing a single 768-dimensional vector for each question or context.

Relevant contexts are then retrieved by selecting those whose embeddings have the highest cosine similarity to the embedding of the input question.

In [32]:
# example

device = "cuda"
from transformers import BertTokenizer, BertModel

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased').to(device)

inputs = tokenizer("This is a sample sentence.", return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
with torch.no_grad():
    outputs = model(**inputs)
    hidden_states = outputs.last_hidden_state
    embedding = torch.mean(hidden_states, dim=1)  # (batch_size=1, embedding size =768)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [33]:
embedding.shape

torch.Size([1, 768])

## Encoding the Candidate Contexts

To perform dense retrieval, every candidate context must first be converted into its corresponding BERT embedding. Since these embeddings are independent of the questions, they only need to be computed once.

Each context is tokenized, passed through the pretrained BERT model, and represented by the mean of its final hidden states, resulting in a 768-dimensional embedding. The embeddings are then stacked into a single tensor of shape `(19035, 768)`, where each row corresponds to one candidate context.

Precomputing and storing these embeddings significantly reduces computation during retrieval, as only the question embedding must be computed at inference time before comparing it against the cached context embeddings using cosine similarity.

In [38]:
embedding_list = []

with torch.no_grad():
    for context in tqdm(candidate_contexts, desc="Encoding candidate contexts"):
        inputs = tokenizer(
            context,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512,
        ).to(device)

        outputs = model(**inputs)

        hidden_states = outputs.last_hidden_state

        embedding = hidden_states.mean(dim=1).squeeze(0).cpu()

        embedding_list.append(embedding)

context_embeddings = torch.stack(embedding_list)

print(context_embeddings.shape)

Encoding candidate contexts:   0%|          | 0/19035 [00:00<?, ?it/s]

torch.Size([19035, 768])


In [42]:
torch.save(context_embeddings,"/content/drive/MyDrive/rag_qa_evaluation/context_embeddings.pt")

Similarly we encode each question and stack the embeddings together into a single (250, 768) pytorch tensor that we can save to disk and reload as needed.

In [40]:
embedding_list = []

with torch.no_grad():
    for qa_item in tqdm(evaluation_benchmark["qas"], desc="Encoding questions"):
        question = qa_item["question"]

        inputs = tokenizer(
            question,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512,
        ).to(device)

        outputs = model(**inputs)

        hidden_states = outputs.last_hidden_state

        embedding = hidden_states.mean(dim=1).squeeze(0).cpu()

        embedding_list.append(embedding)

question_embeddings = torch.stack(embedding_list)

print(question_embeddings.shape)

Encoding questions:   0%|          | 0/250 [00:00<?, ?it/s]

torch.Size([250, 768])


In [43]:
torch.save(question_embeddings, "/content/drive/MyDrive/rag_qa_evaluation/question_embeddings.pt")

### Similarity Retriever

In [44]:
context_embeddings = torch.load("/content/drive/MyDrive/rag_qa_evaluation/context_embeddings.pt")

question_embeddings = torch.load("/content/drive/MyDrive/rag_qa_evaluation/question_embeddings.pt")

## Dense Retrieval with Cosine Similarity

With embeddings available for both the questions and candidate contexts, retrieval can now be performed by comparing their vector representations.

For a given question, cosine similarity is calculated between its embedding and every context embedding. The contexts with the highest similarity scores are treated as the most semantically relevant and returned as the top-*k* retrieval results.

Because the rows of `context_embeddings` follow the same order as `contexts`, the indices of the highest similarity scores can be used directly to recover the corresponding context strings.

In [45]:
def retrieve_cosine(
    question_emb,
    contexts,
    context_embeddings,
    top_k=5
):
    # Ensure the question has shape (1, 768)
    if question_emb.dim() == 1:
        question_emb = question_emb.unsqueeze(0)

    # Keep both tensors on the same device
    question_emb = question_emb.to(context_embeddings.device)

    # Compare the question against every context embedding
    similarities = F.cosine_similarity(
        context_embeddings,
        question_emb,
        dim=1
    )

    # Get the indices of the top-k most similar contexts
    top_indices = torch.topk(
        similarities,
        k=top_k
    ).indices

    return [
        contexts[index]
        for index in top_indices.cpu().tolist()
    ]

In [46]:
retrieve_cosine(question_embeddings[0], candidate_contexts, context_embeddings)

['Shih-Shan Henry Tsai writes that the Yongle Emperor sent his eunuch Yang Sanbao into Tibet in 1413 to gain the allegiance of various Tibetan princes, while the Yongle Emperor paid a small fortune in return gifts for tributes in order to maintain the loyalty of neighboring vassal states such as Nepal and Tibet. However, Van Praag states that Tibetan rulers upheld their own separate relations with the kingdoms of Nepal and Kashmir, and at times "engaged in armed confrontation with them."',
 'Laird writes that the Ming appointed titles to eastern Tibetan princes, and that "these alliances with eastern Tibetan principalities are the evidence China now produces for its assertion that the Ming ruled Tibet," despite the fact that the Ming did not send an army to replace the Mongols after they left Tibet. Yiu Yung-chin states that the furthest western extent of the Ming dynasty\'s territory was Gansu, Sichuan, and Yunnan while "the Ming did not possess Tibet."',
 'During the Five Dynasties a

## Adding Dense Retrieval Results to the QA Dataset

The dense retriever is now applied to every question in the evaluation benchmark. Each question embedding is compared against the complete set of context embeddings, and the top-*k* most similar contexts are selected.

The retrieved passages are stored in the `rag_contexts` field of a copied QA item. Because the question embeddings follow the same order as `qa_items`, the embedding at index `i` corresponds to the question in `qa_items[i]`.

In [69]:
def add_rag_context(
    qa_items,
    contexts,
    retriever,
    question_embeddings,
    context_embeddings,
    top_k=5
):
    result_items = copy.deepcopy(qa_items)

    for i, inst in enumerate(
        tqdm(result_items, desc="Retrieving contexts")
    ):
        question_emb = question_embeddings[i].unsqueeze(0)

        retrieved_contexts = retriever(
            question_emb,
            contexts,
            context_embeddings,
            top_k
        )

        inst["rag_contexts"] = retrieved_contexts

    return result_items

In [48]:
rag_qa_items = add_rag_context(evaluation_benchmark['qas'], candidate_contexts, retrieve_cosine, question_embeddings, context_embeddings)

Retrieving contexts:   0%|          | 0/250 [00:00<?, ?it/s]

In [ ]:
rag_qa_items[0]

In [49]:
evaluate_retriever(rag_qa_items)

0.416

In [50]:
result = evaluate_qa(rag_qa, rag_qa_items)
present_results(result)

Evaluating QA instances:   0%|          | 0/250 [00:00<?, ?it/s]

 Evaluation Results:
Exact Match: 18.00%
F1 Score: 28.97%
ROUGE2 F1: 14.00%
Question: Which which kingdoms did the taifas establish diplomatic relations?
Gold Answer: the Christian kingdoms of the north
Predicted Answer: unknown
Exact Match: 0, F1 Score: 0.0
ROUGE-2 F1-score: 0.0
----------------------------------------
Question: When was Nanjing extended?
Gold Answer: 211 AD
Predicted Answer: 1916
Exact Match: 0, F1 Score: 0.0
ROUGE-2 F1-score: 0.0
----------------------------------------
Question: In what district of Manhattan were the Occupy Wall Street protests?
Gold Answer: Financial District
Predicted Answer: Financial District of Lower Manhattan
Exact Match: 0, F1 Score: 0.5714285714285715
ROUGE-2 F1-score: 0.4
----------------------------------------
Question: Illyrian pirates caused two wars in the Balkans with what Empire?
Gold Answer: Roman
Predicted Answer: Illyria (the First and Second Illyrian Wars)
Exact Match: 0, F1 Score: 0.0
ROUGE-2 F1-score: 0.0
---------------------

The token-overlap retriever outperformed the BERT-based dense retriever across all three answer-quality metrics. Although dense retrieval can capture semantic similarity beyond exact word matches, the pretrained BERT model used here was not specifically optimized for sentence-level similarity or passage retrieval. In contrast, many benchmark questions share distinctive terms with their source contexts, allowing the lexical retriever to identify relevant passages effectively. These results demonstrate that denser representations do not necessarily improve retrieval unless the embedding model is trained for the retrieval objective.

## Part 6 - Experiments

For the overlap and dense retrievers (from part 5 and 6), let's explore what happens when we change the number of retrieved contexts? Present a table of results for k=1, k=5 (already done), k=10, and k=20.


In [70]:
dense_rag_qa_pairs_top20 = add_rag_context(
    evaluation_benchmark["qas"],
    candidate_contexts,
    retrieve_cosine,
    question_embeddings,
    context_embeddings,
    top_k=20
)

Retrieving contexts:   0%|          | 0/250 [00:00<?, ?it/s]

In [72]:
dense_datasets = {
    k: select_top_k_contexts(
        dense_rag_qa_pairs_top20,
        top_k=k
    )
    for k in [1, 5, 10, 20]
}

In [ ]:
k_values_to_run = [1, 10, 20]

overlap_experiment_results = {
    5: rag_overlap_eval
}

dense_experiment_results = {
    5: result
}

for k in k_values_to_run:
    print(f"Evaluating overlap retrieval with k={k}")

    overlap_experiment_results[k] = evaluate_qa(
        rag_qa,
        overlap_datasets[k]
    )

    torch.save(
        overlap_experiment_results,
        "/content/drive/MyDrive/rag_qa_evaluation/overlap_experiment_results.pt"
    )

    print(f"Evaluating dense retrieval with k={k}")

    dense_experiment_results[k] = evaluate_qa(
        rag_qa,
        dense_datasets[k]
    )

    torch.save(
        dense_experiment_results,
        "/content/drive/MyDrive/rag_qa_evaluation/dense_experiment_results.pt"
    )

In [77]:
overlap_experiment_results = torch.load(
    "/content/drive/MyDrive/rag_qa_evaluation/overlap_experiment_results.pt",
    weights_only=False
)

dense_experiment_results = torch.load(
    "/content/drive/MyDrive/rag_qa_evaluation/dense_experiment_results.pt",
    weights_only=False
)

In [91]:
import pandas as pd

rows = []

for retriever_name, experiment_results in [
    ("Token Overlap", overlap_experiment_results),
    ("Dense", dense_experiment_results),
]:
    for k in [1, 5, 10]:
        results = experiment_results[k]

        exact_match = sum(
            item["exact_match"]
            for item in results
        ) / len(results)

        f1_score = sum(
            item["f1_score"]
            for item in results
        ) / len(results)

        rouge2_f1 = sum(
            item["rouge2_f1"]
            for item in results
        ) / len(results)

        rows.append({
            "Retriever": retriever_name,
            "k": k,
            "Exact Match (%)": exact_match * 100,
            "F1 Score (%)": f1_score * 100,
            "ROUGE-2 F1 (%)": rouge2_f1 * 100,
        })

results_table = pd.DataFrame(rows).round(2)

results_table

,Retriever,k,Exact Match (%),F1 Score (%),ROUGE-2 F1 (%)
0,Token Overlap,1,22.4,31.09,16.37
1,Token Overlap,5,27.2,37.40,20.00
2,Token Overlap,10,26.8,38.48,20.37
3,Dense,1,13.6,21.19,10.86
4,Dense,5,18.0,28.97,14.00
5,Dense,10,23.6,33.77,16.67


In [85]:
if hasattr(pipe.model.config, "max_position_embeddings"):
    print("Model max positions:", pipe.model.config.max_position_embeddings)

Model max positions: 4096


In [81]:
def prompt_length(qa_item):
    question = qa_item["question"]
    retrieved_contexts = qa_item["rag_contexts"]

    context = "\n\n".join(
        f"Context {i + 1}:\n{retrieved_context}"
        for i, retrieved_context in enumerate(retrieved_contexts)
    )

    prompt = f"""Answer the question using only the retrieved contexts below.

Return only the shortest answer phrase.
Do not explain.
Do not repeat the question.
If the answer is not contained in the contexts, return "Unknown".

Retrieved contexts:
{context}

Question: {question}

Answer:"""

    return len(pipe.tokenizer(prompt)["input_ids"])

In [82]:
print("k=10:",
      prompt_length(overlap_datasets[10][0]))

print("k=20:",
      prompt_length(overlap_datasets[20][0]))

k=10: 2514
k=20: 4638


### Discussion

The effect of increasing the number of retrieved contexts differs between the two retrieval methods.

For the token-overlap retriever, increasing the number of retrieved contexts from \(k=1\) to \(k=5\) produces a noticeable improvement across all evaluation metrics. However, increasing the retrieval depth further to \(k=10\) provides only marginal gains in F1 and ROUGE-2 while Exact Match remains essentially unchanged. This suggests that the lexical retriever usually ranks the relevant passage within its top five retrieved contexts, so additional passages contribute little useful information and may instead introduce noise.

The dense retriever exhibits a different behavior. Although its overall performance remains below that of the token-overlap retriever, all three metrics improve consistently as \(k\) increases. This indicates that the dense retriever often retrieves relevant passages lower in its ranking, allowing larger values of \(k\) to recover additional useful context for answer generation.

The original experiment specification also included \(k=20\). During experimentation, we measured the prompt lengths generated by our RAG pipeline and observed that prompts containing twenty retrieved contexts exceeded the model's maximum input length. The model supports a maximum context window of 4096 tokens, while a representative prompt with \(k=20\) contained approximately 4638 input tokens. Consequently, the full retrieved context could not be processed without truncation, making comparisons with smaller values of \(k\) unreliable. Additionally, inference time increased substantially as the prompt length grew, with evaluation at \(k=10\) already requiring significantly longer runtimes than smaller values of \(k\). For these reasons, the analysis was limited to \(k=1\), \(k=5\), and \(k=10\).

# Part 7 — Improving the QA System with Reciprocal Rank Fusion

The previous experiments showed that lexical and dense retrieval behave differently.

The token-overlap retriever performs well when the question and relevant passage share distinctive terms. The dense retriever can capture broader semantic similarity, but its ranking is less reliable because the underlying BERT embeddings were not trained specifically for passage retrieval.

Rather than selecting one retriever, we combine both rankings using **Reciprocal Rank Fusion (RRF)**.

RRF assigns each retrieved context a score based only on its rank:

$$
\operatorname{RRF}(d)
=
\sum_{r \in R}
\frac{1}{c + \operatorname{rank}_r(d)}
$$


where:

- $d$ is a candidate context;
- $R$ is the set of retrieval systems;
- $\operatorname{rank}_r(d)$ is the rank assigned to context $d$ by retriever $r$;
- $c$ is a constant that limits the influence of very small rank differences.

In this experiment, \(c=60\), a commonly used default.

RRF avoids directly combining lexical-overlap and cosine-similarity scores, which are measured on incompatible scales. Instead, it rewards contexts that are ranked highly by either retriever, with additional weight given to contexts that both retrievers rank highly.

The resulting pipeline is:

```text
Question
   │
   ├──► Token-Overlap Retriever
   │
   └──► Dense Retriever
              │
              ▼
      Reciprocal Rank Fusion
              │
              ▼
       Top-k Fused Contexts
              │
              ▼
             LLM
              │
              ▼
            Answer

In [93]:
from collections import defaultdict

def reciprocal_rank_fusion(
    lexical_contexts,
    dense_contexts,
    top_k=5,
    rrf_constant=60
):

    fused_scores = defaultdict(float)

    for rank, context in enumerate(
        lexical_contexts,
        start=1
    ):
        fused_scores[context] += (
            1 / (rrf_constant + rank)
        )

    for rank, context in enumerate(
        dense_contexts,
        start=1
    ):
        fused_scores[context] += (
            1 / (rrf_constant + rank)
        )

    ranked_contexts = sorted(
        fused_scores.items(),
        key=lambda item: item[1],
        reverse=True
    )

    return [
        context
        for context, _ in ranked_contexts[:top_k]
    ]

In [94]:
def build_rrf_qa_pairs(
    lexical_qa_items,
    dense_qa_items,
    top_k=5,
    rrf_constant=60
):
    """
    Create QA items containing contexts ranked by RRF.
    """
    result_items = copy.deepcopy(lexical_qa_items)

    for i, item in enumerate(result_items):
        lexical_contexts = (
            lexical_qa_items[i]["rag_contexts"]
        )

        dense_contexts = (
            dense_qa_items[i]["rag_contexts"]
        )

        item["rag_contexts"] = reciprocal_rank_fusion(
            lexical_contexts=lexical_contexts,
            dense_contexts=dense_contexts,
            top_k=top_k,
            rrf_constant=rrf_constant
        )

    return result_items

In [95]:
rrf_rag_qa_pairs = build_rrf_qa_pairs(
    lexical_qa_items=overlap_rag_qa_pairs_top20,
    dense_qa_items=dense_rag_qa_pairs_top20,
    top_k=5,
    rrf_constant=60
)

In [97]:
rrf_evaluation_results = evaluate_qa(
    rag_qa,
    rrf_rag_qa_pairs
)

present_results(
    rrf_evaluation_results
)

Evaluating QA instances:   0%|          | 0/250 [00:00<?, ?it/s]

 Evaluation Results:
Exact Match: 32.80%
F1 Score: 45.25%
ROUGE2 F1: 22.90%
Question: Which which kingdoms did the taifas establish diplomatic relations?
Gold Answer: the Christian kingdoms of the north
Predicted Answer: Unknown
Exact Match: 0, F1 Score: 0.0
ROUGE-2 F1-score: 0.0
----------------------------------------
Question: When was Nanjing extended?
Gold Answer: 211 AD
Predicted Answer: Nanjing, China
Exact Match: 0, F1 Score: 0.0
ROUGE-2 F1-score: 0.0
----------------------------------------
Question: In what district of Manhattan were the Occupy Wall Street protests?
Gold Answer: Financial District
Predicted Answer: Lower Manhattan
Exact Match: 0, F1 Score: 0.0
ROUGE-2 F1-score: 0.0
----------------------------------------
Question: Illyrian pirates caused two wars in the Balkans with what Empire?
Gold Answer: Roman
Predicted Answer: Rome
Exact Match: 0, F1 Score: 0.0
ROUGE-2 F1-score: 0.0
----------------------------------------
Question: How many people were killed in the la

In [98]:
comparison_table = pd.DataFrame({
    "Retriever": [
        "Token Overlap (k=5)",
        "Dense (k=10)",
        "Reciprocal Rank Fusion"
    ],
    "Exact Match (%)": [
        27.20,
        23.60,
        32.80
    ],
    "F1 Score (%)": [
        37.40,
        33.77,
        45.25
    ],
    "ROUGE-2 F1 (%)": [
        20.00,
        16.67,
        22.90
    ]
})

comparison_table

,Retriever,Exact Match (%),F1 Score (%),ROUGE-2 F1 (%)
0,Token Overlap (k=5),27.2,37.40,20.00
1,Dense (k=10),23.6,33.77,16.67
2,Reciprocal Rank Fusion,32.8,45.25,22.90


### Discussion

The Reciprocal Rank Fusion (RRF) retriever achieved the strongest overall performance among all retrieval methods evaluated in this project.

| Retriever | Exact Match | F1 | ROUGE-2 F1 |
|-----------|------------:|---:|-----------:|
| Token Overlap (k=5) | 27.2 | 37.4 | 20.0 |
| Dense (k=10) | 23.6 | 33.8 | 16.7 |
| **Reciprocal Rank Fusion** | **32.8** | **45.3** | **22.9** |

Unlike the previous experiments, which relied on either lexical or dense retrieval independently, RRF combines the ranked outputs of both retrievers before selecting the final contexts provided to the language model.

The earlier experiments demonstrated that the two retrieval strategies exhibit complementary behavior. The lexical retriever consistently ranked passages containing strong lexical matches near the top of the list, while the dense retriever often recovered semantically relevant passages that appeared lower in the lexical ranking. Reciprocal Rank Fusion exploits this complementarity by rewarding contexts that are ranked highly by either retriever, while giving the greatest weight to contexts that both retrieval systems identify as relevant.

This simple rank-based fusion strategy produced substantial improvements across all evaluation metrics. Compared with the best lexical retriever, Exact Match increased from **27.2%** to **32.8%**, F1 Score improved from **37.4%** to **45.3%**, and ROUGE-2 F1 increased from **20.0%** to **22.9%**.

These results suggest that combining complementary retrieval signals is more effective than relying on either lexical matching or semantic similarity alone. Furthermore, Reciprocal Rank Fusion achieves these improvements without retraining the retrieval models or introducing additional learnable parameters, making it a simple yet powerful enhancement to the retrieval-augmented generation pipeline.